### Attention Mechanism

### Attention scores

- The attention score efficient by Q · Kᵀ.
- From the previous section we can imagine our tensors `Q` and `K` of the shape `(T, C/h)` so `T` tokens of `C/h` dimensions.
- We must transpose K for dimensionality matching.
- Now in reality out shape is `(B, h, T, C/h)` which is done **per batch, per head simultaneously**, so each of the `h` heads will produce their own `(T,T)` attention matrix providing `h` independent views of how tokens relate.
- The matmul operation is `torch.matmul` or `@` specifically ``scores = q @ k.transpose(-2, -1)``
- The negative indicies count from the end.

In [37]:
import torch
import torch.nn as nn
import math

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, block_size, n_embed, n_head):
        super().__init__()
        self.n_embed = n_embed
        self.n_head = n_head
        self.wte = nn.Embedding(vocab_size, n_embed)
        self.wpe = nn.Embedding(block_size, n_embed)
        self.ln = nn.LayerNorm(n_embed)
        self.qkv = nn.Linear(n_embed, 3*n_embed)

    # This function will create:
    #  - Token Embed 
    #  - Positional Embedding 
    #  - Merge the two embedding 
    #  - Layer Normalise 
    # Question 1: In the full implementation is the input actually the same (i.e. the raw prompt) for the positional Embedding and the Token embedding
    def process_raw_input(self,idx):
        print("start process_raw_input()")
        token_embed = self.wte(idx)
        B, T = idx.shape
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        # Answer 1: We only require the matching dimension and not any data so no need to worry about this
        x_unormed = token_embed + pos_emb
        x_norm = self.ln(x_unormed)
        print("end process_raw_input()")
        return x_norm

    def attention_preproces(self, t):
        print("start attention_preprocess()")
        B, T, C = t.shape
        h = self.n_head
        t_reviewed = t.view(B, T, h, C//h)
        t_transposed = t_reviewed.transpose(1, 2)
        print("end attention_preprocess()")
        return t_transposed

    def attention_head(self, q, k, v):
        # Matmul
        print("start attention_head()")
        attn_score = q @ k.transpose(-1, -2)
        # scale
        # Question 2: I wonder if this multiplication is element wise and if so does this mean that '*' is overloaded
        # Question 3: I wonder if this even correct at all 😂
        # Answer 3: This didn't work, mixing between standard python data types and pytroch needs to be taken with care
        d_k = q.shape[-1]
        attn_score_scaled = attn_score * 1/math.sqrt(d_k)
        # MASK 
        T = attn_score_scaled.shape(-1)
        mask = torch.tril(torch.ones(T, T))
        masked = attn_score_scaled.masked_fill(mask == 0, float('-inf'))
        print("end attention_head()")

        return attn_score_scaled

    def forward(self, prompts):
        print("Start forward()")
        x = self.process_raw_input(prompts)
        print("passed process_raw_input()")
        qkv = self.qkv(x)
        B, T, C = qkv.shape
        h = self.n_head
        q_raw, k_raw, v_raw = qkv.split(self.n_embed, dim=-1)
        q = self.attention_preproces(q_raw)
        k = self.attention_preproces(k_raw)
        v = self.attention_preproces(v_raw)
        scaled_attn = self.attention_head(q, k, v)
        return scaled_attn

In [39]:
attn = Transformer(vocab_size = 1, block_size=4, n_embed=4, n_head=2)
x = torch.tensor([[0,0]])
print(x.shape)
s = attn(x)
print(s)
print(s.shape)   # torch.Size([1, 2, 2, 2]) = (B, h, T, T)

torch.Size([1, 2])
Start forward()
start process_raw_input()
end process_raw_input()
passed process_raw_input()
start attention_preprocess()
end attention_preprocess()
start attention_preprocess()
end attention_preprocess()
start attention_preprocess()
end attention_preprocess()
start attention_head()
end attention_head()
tensor([[[[ 0.6610,  0.4098],
          [ 0.3232,  0.2033]],

         [[ 0.0899, -0.0068],
          [-0.0627, -0.0402]]]], grad_fn=<DivBackward0>)
torch.Size([1, 2, 2, 2])


## Scale operation

- Given the raw scores the we produced above, scale the values by `√(C/h)`
- The head dimension is of size `C/h`, each score is a dot product over this size, what is possible to occur is either an exploding or diminishing of these scores due to the sequence of multiplication, which can later affect the `softmax` score thus affect the learning process of the model.
- Something to keep in mind to use the `.shape` or `.shape[index]`, to ensure the dimensions track based on the architecture design

## Masking
- This is the operation that ensures the model doesn't attend to the future tokens, this is done by replacing given values to future tokens with -∞
- Transformer models of this kind are autoregressive meaning for some token `j`, the model will try to predict `j+1` using tokens `1 -> j`.
- Note that -∞ is specifically chosen based on it's interpretation on the `softmax` function. 
- so our attention matrix essentialy becomes lower triangular matrix where the top half are masked and the bottom are left as is.
- The `torch` function used here is: `mask = torch.tril(torch.ones(T, T))` **together** with the ``masked_fill(condition, value)``
- `condition` means which value you desire to be masked
- `value` means what's the masking value of choice 
- `masked = scaled.masked_fill(mask == 0, float('-inf'))`

## Softmax
- After masking we finally convert these scores into probabilities using the softmax function.
- The function used is ``torch.softmax(masked_scores, dim=-1)``
  - ``dim=-1`` indicates over which dimension the softmax should sum over, keep in mind we wish to sum over the token.

**Implementation Note**
You may observe multiple forms of the `softmax` being called:
1. ``weights = torch.softmax(masked_scores, dim=-1) function on torch``
2. ``weights = masked_scores.softmax(dim=-1) Tensor method (same thing)``
3. ``weights = F.softmax(masked_scores, dim=-1) From torch.nn.Function (import F)``

## Final Matmul
- The final operation produces a weight based on the value of the token with a probability distribution of the attention

In [46]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, n_embed, block_size, n_heads):
        super().__init__()
        self.n_embd = n_embed
        self.n_head = n_heads
        self.wte = nn.Embedding(vocab_size, n_embed)
        self.wpe = nn.Embedding(block_size, n_embed)
        self.ln = nn.LayerNorm(n_embed)
        self.qkv = nn.Linear(n_embed, 3*n_embed)

    def process_data(self, t):
        token_emb = self.wte(t)
        B, T = t.shape
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        x_merged = token_emb + pos_emb
        x_norm = self.ln(x_merged)
        return x_norm

    def preprocess_attention(self, t):
        B, T, C = t.shape
        h = self.n_head
        t_reviewed = t.view(B, T, h, C//h)
        t_transposed = t_reviewed.transpose(1, 2)
        return t_transposed

    def attention_mech(self, q, k, v):
        T = q.shape[-2]
        scores = q @ k.transpose(-1, -2)
        scores_scaled = scores / (self.n_head ** 0.5)
        mask = torch.tril(torch.ones(T, T))
        score_masked = scores_scaled.masked_fill(mask == 0, float('-inf'))
        probs = score_masked.softmax(dim=-1)
        output = probs @ v
        return output


    def forward(self, idx):
        x = self.process_data(idx)
        qkv = self.qkv(x)
        q_raw, k_raw, v_raw = qkv.split(self.n_embd, dim=-1)
        q = self.preprocess_attention(q_raw)
        k = self.preprocess_attention(k_raw)
        v = self.preprocess_attention(v_raw)
        output = self.attention_mech(q, k, v)
        return output






In [47]:
model = Transformer(vocab_size=6, n_embed=4, block_size=4, n_heads=2)
idx = torch.tensor([[0, 1, 2, 3]])
out = model(idx)
print(out.shape)   # torch.Size([1, 4, 2, 2]) = (B, h, T, C/h)

torch.Size([1, 2, 4, 2])
